# ALGORYTMY I STRUKTURY DANYCH - PROJEKT nr #7 - *Bajtoladnia*

Najpierw import paketów i bibliotek

In [ ]:
import random
import heapq
from typing import List, Tuple, Optional

### Tworzymy klasę pomocniczą **Graph**

In [ ]:
class Graph:
    """
    Reprezentacja grafu za pomocą list incydencji
    Wierzchołki numerujemy 0..n-1.
    Każdy wpis w adj[v] to para (u, waga)
    """
    def __init__(self, n_vertices: int):
        self.n = n_vertices
        self.adj: List[List[Tuple[int, int]]] = [[] for _ in range(n_vertices)]

    def add_edge(self, u: int, v: int, w: int, undirected: bool = True) -> None:
        """Dodaje krawędź u -> v o wadze w. Dla grafu nieskierowanego dodaje też v -> u"""
        self.adj[u].append((v, w))
        if undirected:
            self.adj[v].append((u, w))

### Klasa główna do tworzenia planszy - **Bajtolandia**

In [ ]:
class Bajtolandia:
    """
    Klasa opisująca zadanie 'Bajtolandia'.
    Wizualizacja: (0, 0) znajduje się w LEWYM DOLNYM rogu.
    """

    def __init__(
        self,
        n: int = None,
        m: int = None,
        mapa_kosztow: List[List[int]] = None,
        start_pos: Tuple[int, int] = None,
        end_pos: Tuple[int, int] = None,
        n_min: int = 1, n_max: int = 50,
        m_min: int = 1, m_max: int = 50,
        min_cost: int = 1, max_cost: int = 9,
    ):
        """
        Konstruktor klasy
        """


        # --- 1. Ustalanie wymiarów i kosztów ---
        if mapa_kosztow is not None:
            self.n = len(mapa_kosztow)
            self.m = len(mapa_kosztow[0]) if self.n > 0 else 0
            self.cost = mapa_kosztow
            flat = [x for row in mapa_kosztow for x in row]
            self.min_cost = min(flat) if flat else 0
            self.max_cost = max(flat) if flat else 0
        elif n is not None and m is not None:
            self.n = n
            self.m = m
            self.min_cost = min_cost
            self.max_cost = max_cost
            self.cost = [[random.randint(min_cost, max_cost) for _ in range(m)] for _ in range(n)]
        else:
            self.n = random.randint(n_min, n_max)
            self.m = random.randint(m_min, m_max)
            self.min_cost = min_cost
            self.max_cost = max_cost
            self.cost = [[random.randint(min_cost, max_cost) for _ in range(self.m)] for _ in range(self.n)]

        # --- 2. Walidacja miast ---
        self.start = start_pos
        self.end = end_pos
        self.graph = None

        if self.start is not None and self.end is not None:
            sx, sy = self.start
            tx, ty = self.end
            
            # Walidacja granic
            if not (0 <= sx < self.n and 0 <= sy < self.m):
                raise ValueError(f"Start {self.start} poza planszą (max x={self.n-1}, y={self.m-1})")
            if not (0 <= tx < self.n and 0 <= ty < self.m):
                raise ValueError(f"Cel {self.end} poza planszą (max x={self.n-1}, y={self.m-1})")
            
            # Walidacja sąsiedztwa
            manhattan = abs(sx - tx) + abs(sy - ty)
            if manhattan == 0:
                raise ValueError("Miasta są w tym samym punkcie!")
            if manhattan == 1:
                raise ValueError("Miasta sąsiadują ze sobą (muszą być oddalone)!")

    # --- Logika bez zmian ---
    def _cell_id(self, i: int, j: int) -> int: return i * self.m + j
    def _coords_from_id(self, vid: int) -> Tuple[int, int]: return divmod(vid, self.m)

    def losuj_miasta(self) -> None:
        """
        Funkcja służąca do losowania miast gdy nie są podane
        """


        if self.start and self.end: return
        while True:
            sx, sy = random.randint(0, self.n - 1), random.randint(0, self.m - 1)
            tx, ty = random.randint(0, self.n - 1), random.randint(0, self.m - 1)
            if abs(sx - tx) + abs(sy - ty) > 1:
                self.start = (sx, sy)
                self.end = (tx, ty)
                break

    def build_graph(self) -> None:
        """
        Tworzenie grafu do problemu
        """
        self.graph = Graph(self.n * self.m)
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for i in range(self.n):
            for j in range(self.m):
                u = self._cell_id(i, j)
                for dx, dy in dirs:
                    ni, nj = i + dx, j + dy
                    if 0 <= ni < self.n and 0 <= nj < self.m:
                        self.graph.add_edge(u, self._cell_id(ni, nj), self.cost[ni][nj])

    def dijkstra(self, start_id: int):
        """
        Algorytm Dikstry
        """
        if not self.graph: raise ValueError("Zbuduj graf!")
        n = self.graph.n
        dist = [10**18] * n
        prev = [-1] * n
        pq = []
        
        dist[start_id] = self._start_cost(start_id)
        heapq.heappush(pq, (dist[start_id], start_id))
        
        while pq:
            d, v = heapq.heappop(pq)
            if d > dist[v]: continue
            for neigh, w in self.graph.adj[v]:
                if d + w < dist[neigh]:
                    dist[neigh] = d + w
                    prev[neigh] = v
                    heapq.heappush(pq, (dist[neigh], neigh))
        return dist, prev

    def _start_cost(self, sid: int) -> int:
        r, c = self._coords_from_id(sid)
        return self.cost[r][c]

    def znajdz_najtansza_trase(self):
        if not self.start: self.losuj_miasta()
        if not self.graph: self.build_graph()
        
        sid = self._cell_id(*self.start)
        tid = self._cell_id(*self.end)
        dist, prev = self.dijkstra(sid)
        
        if dist[tid] >= 10**18: return [], -1
        
        path = []
        curr = tid
        while curr != -1:
            path.append(curr)
            curr = prev[curr]
        return [self._coords_from_id(v) for v in reversed(path)], dist[tid]

    # --- WIZUALIZACJA (Poprawiona: (0,0) na dole) ---

    def wypisz_koszty(self):
        print(f"Rozmiar: {self.n} (wiersze) x {self.m} (kolumny)")
        print("Mapa kosztów (wiersz 0 na dole):")
        print("-" * (self.m * 3 + 2))
        
        # ### ZMIANA: Iterujemy od ostatniego wiersza do 0
        for i in range(self.n - 1, -1, -1):
            row_str = " ".join(f"{self.cost[i][j]:2}" for j in range(self.m))
            print(f"| {row_str} |  (wiersz {i})")
            
        print("-" * (self.m * 3 + 2))
        # Oś X (kolumny)
        cols_idx = " ".join(f"{j:2}" for j in range(self.m))
        print(f"  {cols_idx}    (kolumny)")
        print()

    def wypisz_miasta(self):
        print(f"Start (Bajtogóra): {self.start}")
        print(f"Cel   (Bitogóra):  {self.end}")

    def wypisz_trase(self, path, cost):
        if not path:
            print("Brak trasy!")
            return
        print(f"Minimalny koszt: {cost}")
        print(f"Trasa: {path}")

    def rysuj_trase_na_planszy(self, path):
        # Tworzymy pustą siatkę
        grid = [['.' for _ in range(self.m)] for _ in range(self.n)]
        
        # Naniesienie trasy
        for r, c in path:
            grid[r][c] = '#'
            
        # Naniesienie miast
        if self.start: grid[self.start[0]][self.start[1]] = 'S'
        if self.end: grid[self.end[0]][self.end[1]] = 'T'
        
        print("Wizualizacja trasy (0,0 na dole):")
        print("-" * (self.m * 2 + 3))
        
        # ### ZMIANA: Wypisujemy od wiersza n-1 do 0
        for i in range(self.n - 1, -1, -1):
            row_str = " ".join(grid[i])
            print(f"| {row_str} | {i}")
            
        print("-" * (self.m * 2 + 3))
        # Oś X
        print("  " + " ".join(str(j % 10) for j in range(self.m)))
        print()

### Przykładowe wywołanie

In [ ]:
if __name__ == "__main__":
    # Definiujemy własną mapę (np. ściana kosztów 9 z jednym przejściem 1)
    moja_mapa = [
        [1, 1, 1, 9, 9],
        [9, 9, 1, 9, 9],
        [9, 9, 1, 1, 1],
        [9, 9, 9, 9, 1],
        [1, 1, 1, 1, 1]
    ]
    
    # Tworzymy obiekt podając mapę i miasta
    # Uwaga: współrzędne w Pythonie liczymy od 0!
    # (0, 0) to lewy górny róg
    baj = Bajtolandia(
        mapa_kosztow=moja_mapa,
        start_pos=(1, 2),  # Bajtogóra
        end_pos=(0, 0)     # Bitogóra
    )
    
    # Nie musimy już wołać losuj_miasta ani build_graph ręcznie,
    # znajdz_najtansza_trase zrobi to za nas jeśli trzeba,
    # ale dla porządku można:
    baj.build_graph()
    
    baj.wypisz_koszty()
    baj.wypisz_miasta()
    
    path, cost = baj.znajdz_najtansza_trase()
    baj.wypisz_trase(path, cost)
    baj.rysuj_trase_na_planszy(path)

Rozmiar: 5 (wiersze) x 5 (kolumny)
Mapa kosztów (wiersz 0 na dole):
-----------------
|  1  1  1  1  1 |  (wiersz 4)
|  9  9  9  9  1 |  (wiersz 3)
|  9  9  1  1  1 |  (wiersz 2)
|  9  9  1  9  9 |  (wiersz 1)
|  1  1  1  9  9 |  (wiersz 0)
-----------------
   0  1  2  3  4    (kolumny)

Start (Bajtogóra): (1, 2)
Cel   (Bitogóra):  (0, 0)
Minimalny koszt: 4
Trasa: [(1, 2), (0, 2), (0, 1), (0, 0)]
Wizualizacja trasy (0,0 na dole):
-------------
| . . . . . | 4
| . . . . . | 3
| . . . . . | 2
| . . S . . | 1
| T # # . . | 0
-------------
  0 1 2 3 4



In [ ]:
if __name__ == "__main__":
    # Plansza 10x10, losowe koszty, losowe miasta
    baj = Bajtolandia(n=10, m=10)
    
    
    baj.build_graph()
    
    baj.wypisz_koszty()
    baj.wypisz_miasta()
    
    path, cost = baj.znajdz_najtansza_trase()
    baj.wypisz_trase(path, cost)
    baj.rysuj_trase_na_planszy(path)

Rozmiar: 10 (wiersze) x 10 (kolumny)
Mapa kosztów (wiersz 0 na dole):
--------------------------------
|  1  1  7  8  1  6  8  6  1  5 |  (wiersz 9)
|  9  3  6  2  6  5  2  5  3  8 |  (wiersz 8)
|  6  6  4  9  7  5  8  1  9  7 |  (wiersz 7)
|  4  2  8  5  3  3  4  3  4  3 |  (wiersz 6)
|  5  1  3  6  2  9  1  3  7  5 |  (wiersz 5)
|  8  6  3  5  7  2  5  6  8  4 |  (wiersz 4)
|  1  9  6  8  3  1  9  8  6  4 |  (wiersz 3)
|  3  8  2  7  7  7  3  6  1  9 |  (wiersz 2)
|  7  5  6  5  3  4  1  8  2  8 |  (wiersz 1)
|  3  4  9  6  2  5  5  7  3  2 |  (wiersz 0)
--------------------------------
   0  1  2  3  4  5  6  7  8  9    (kolumny)

Start (Bajtogóra): None
Cel   (Bitogóra):  None
Minimalny koszt: 15
Trasa: [(3, 0), (4, 0), (5, 0), (6, 0), (7, 0)]
Wizualizacja trasy (0,0 na dole):
-----------------------
| . . . . . . . . . . | 9
| . . . . . . . . . . | 8
| T . . . . . . . . . | 7
| # . . . . . . . . . | 6
| # . . . . . . . . . | 5
| # . . . . . . . . . | 4
| S . . . . . . . . . | 3
| 

In [ ]:
if __name__ == '__main__':

    # Wszystko losowe (rozmiar od 1 do 50)
    baj = Bajtolandia()
    path, cost = baj.znajdz_najtansza_trase()
    
    baj.build_graph()
    
    baj.wypisz_koszty()
    baj.wypisz_miasta()

    baj.wypisz_trase(path, cost)
    baj.rysuj_trase_na_planszy(path)

Rozmiar: 12 (wiersze) x 33 (kolumny)
Mapa kosztów (wiersz 0 na dole):
-----------------------------------------------------------------------------------------------------
|  3  2  4  1  6  7  6  2  6  8  5  6  1  2  7  8  5  3  4  2  8  6  4  4  5  5  7  4  4  7  2  1  3 |  (wiersz 11)
|  5  4  5  4  3  4  4  5  1  7  3  8  3  7  2  4  8  8  7  8  6  4  1  6  7  1  8  2  2  1  4  6  7 |  (wiersz 10)
|  9  1  6  1  3  3  9  9  1  6  5  1  3  4  8  6  1  7  7  6  6  2  7  8  1  3  7  5  4  5  5  2  9 |  (wiersz 9)
|  5  8  4  9  4  6  7  5  2  2  7  6  8  9  8  4  6  9  9  9  5  3  8  1  8  7  8  4  3  5  1  3  5 |  (wiersz 8)
|  9  7  2  4  9  5  5  2  9  7  4  2  1  8  2  6  7  7  3  2  3  6  4  7  9  6  3  8  1  6  2  5  4 |  (wiersz 7)
|  5  2  5  3  4  9  7  1  6  3  8  3  2  6  5  8  4  2  9  1  8  1  6  2  5  9  7  7  3  4  8  1  4 |  (wiersz 6)
|  7  1  2  9  5  3  5  4  7  2  9  1  7  8  6  3  6  1  3  4  1  2  7  9  4  6  1  4  1  7  9  1  7 |  (wiersz 5)
|  2  5  8  2  5  8  